In [0]:
FEATURE_TABLE_NAME = "workspace.marketing_campaign.gold_customer_features"
PREDICTION_TABLE_NAME = "workspace.marketing_campaign.customer_campaign_predictions"
TABLEAU_TABLE_NAME = "workspace.marketing_campaign.tableau_campaign_dashboard"

features_df = spark.table(FEATURE_TABLE_NAME)
predictions_df = spark.table(PREDICTION_TABLE_NAME)

print(f"Features table: {FEATURE_TABLE_NAME}")
print(f"Predictions table: {PREDICTION_TABLE_NAME}")
print(f"Tableau table: {TABLEAU_TABLE_NAME}")

In [0]:
tableau_df = (
    features_df.alias("f")
    .join(
        predictions_df.select(
            "customer_id",
            "predicted_response",
            "response_probability",
            "prediction_timestamp"
        ).alias("p"),
        on="customer_id",
        how="left"
    )
)

print(f"Rows after join: {tableau_df.count()}")
display(tableau_df.limit(10))

In [0]:
from pyspark.sql.functions import col, when, round

tableau_df = (
    tableau_df
    .withColumn(
        "age_band",
        when(col("customer_age") < 30, "Under 30")
        .when(col("customer_age") < 45, "30-44")
        .when(col("customer_age") < 60, "45-59")
        .otherwise("60+")
    )
    .withColumn(
        "income_band",
        when(col("income") < 30000, "Under 30k")
        .when(col("income") < 60000, "30k-59k")
        .when(col("income") < 90000, "60k-89k")
        .otherwise("90k+")
    )
    .withColumn(
        "spend_band",
        when(col("total_spend") < 500, "Low spend")
        .when(col("total_spend") < 1500, "Medium spend")
        .otherwise("High spend")
    )
    .withColumn(
        "response_probability_pct",
        round(col("response_probability") * 100, 2)
    )
)

In [0]:
tableau_output_df = tableau_df.select(
    "customer_id",
    "education",
    "marital_status",
    "income",
    "income_band",
    "customer_age",
    "age_band",
    "has_children",
    "recency",
    "total_spend",
    "spend_band",
    "total_purchases",
    "numwebvisitsmonth",
    "accepted_previous_campaign",
    "complain",
    col("response").alias("actual_response"),
    "predicted_response",
    "response_probability",
    "response_probability_pct",
    "prediction_timestamp"
)

display(tableau_output_df.limit(10))

In [0]:
(
    tableau_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLEAU_TABLE_NAME)
)

In [0]:
tableau_check_df = spark.table(TABLEAU_TABLE_NAME)

print(f"Rows saved: {tableau_check_df.count()}")
print(f"Columns saved: {len(tableau_check_df.columns)}")

display(tableau_check_df.limit(20))